# Iter-SPIN Healthcare — Step 2 (batch): HSIC + perplexity + medical accuracy, all snapshots vs one baseline

Sweeps every `coupled_iter_health_n*.json` (and any `coupled_control_*_health_n*.json` if present) for the
variant comparison on healthcare. Baseline (no suppression) measured ONCE; each set is zeroed, measured,
restored (reversible, no reload). **HSIC estimator / σ grid / ppl window identical to `SPIN_iter_step2.ipynb`**;
eval data is the healthcare pooled fairness/privacy reps, and capability adds PubMedQA perplexity + medical
accuracy (MedMCQA, MMLU-clinical) as in the healthcare Step 2. This is the decoupling-per-capability-cost
curve — the informative variant comparison (awareness is ceiling/null on healthcare, so Steps 3–4 are
reserved for one chosen point).

**Inputs (Kaggle Dataset inputs):** the iter Step-1 snapshots `coupled_iter_health_n*.json`, and the Step-2
eval splits `eval_fair_health.csv`, `eval_priv_health.csv`. Internet ON. GPU **T4×2**. **Outputs:**
`metrics_iter_health.json` + a matched-budget table.

**Note:** for the full dominance test you also need the matched-budget control sets
(`coupled_control_{global,percat}_health_n*.json`); this notebook scores them automatically if present.
The control generator (healthcare port of `SPIN_iter_control.ipynb`) is a separate notebook — ask if needed.

**Decoupling stat pack (matches the general health Step 2).** Each set is scored with the **full pack**: raw HSIC (final + per-layer, σ grid) plus the norm-robust cross-checks — **normalised HSIC**, **linear CKA**, and **representation-norm diagnostics** (mean ‖·‖ of the fairness/privacy reps) — with paired bootstrap 95% CIs on the HSIC drop. Per-subset HSIC is deliberately omitted from the sweep (extra forward passes per snapshot); measure it once at the chosen operating point instead.

## Step 0 — config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, gc, time, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0','sentencepiece'], check=False)
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

MODEL_ID='Qwen/Qwen2-7B-Instruct'
USE_FAST=False; HSIC_N=300; PPL_SEQLEN=2048; SIGMAS=[50,100,200,400]
USE_CHAT_TEMPLATE=False; DTYPE=torch.float16; BOOT=500; SEED=0
MED_BENCH=True; N_MEDMCQA=1000
MMLU_CLIN=['clinical_knowledge','professional_medicine','college_medicine','anatomy','medical_genetics']
SUBSET_ORDER=['OMAQ','EHAI','FBRT-Manual','FBRT-LLM','TRINDS','CC-Manual','CC-LLM']
np.random.seed(SEED); torch.manual_seed(SEED)
def find(name):
    for r in ['.','/kaggle/input','/kaggle/working']:
        h=glob.glob(os.path.join(r,'**',name),recursive=True)
        if h: return h[0]
    return None
def find_all(name):
    h=[]
    for r in ['.','/kaggle/input','/kaggle/working']: h+=glob.glob(os.path.join(r,'**',name),recursive=True)
    return sorted(set(h))
print('config | model', MODEL_ID, '| sigmas', SIGMAS, '| MED_BENCH', MED_BENCH)

## Step 1 — model + healthcare eval pools + capability corpora

In [ ]:
EF=find('eval_fair_health.csv'); EP=find('eval_priv_health.csv')
assert EF and EP, 'add Step-2 eval splits (eval_fair_health.csv, eval_priv_health.csv) as input'
fair=pd.read_csv(EF); priv=pd.read_csv(EP)
if 'subset' not in fair.columns: fair['subset']='(all)'
def stratified(dfq, n, seed=SEED):
    subs=dfq['subset'].unique(); per=max(1,n//len(subs)); parts=[]
    for s in subs:
        d=dfq[dfq['subset']==s]; parts.append(d.sample(n=min(per,len(d)),random_state=seed))
    out=pd.concat(parts)
    return out.sample(n=n,random_state=seed) if len(out)>n else out
fair_q=stratified(fair,HSIC_N)['question'].tolist()
priv_q=priv['question'].sample(n=min(HSIC_N,len(priv)),random_state=SEED).tolist()

tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
_wt=load_dataset('wikitext','wikitext-2-raw-v1',split='test')['text']
wt_ids=tokenizer('\n\n'.join(_wt), return_tensors='pt').input_ids
med_ids=None
if MED_BENCH:
    try: pq=load_dataset('pubmed_qa','pqa_labeled',split='train')
    except Exception: pq=load_dataset('pubmed_qa','pqa_labeled',split='train',trust_remote_code=True)
    cg=find('cal_general_health.csv'); used=set(pd.read_csv(cg)['prompt'].astype(str)) if cg else set()
    med_ids=tokenizer('\n\n'.join(str(a) for q,a in zip(pq['question'],pq['long_answer']) if str(q) not in used), return_tensors='pt').input_ids
print('model loaded | fair',len(fair_q),'priv',len(priv_q))

## Step 2 — metric functions (HSIC/ppl mirror SPIN_iter_step2; + medical accuracy)

In [ ]:
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def reps_all(queries,bs=8):
    acc=None
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        hs=model(**enc,output_hidden_states=True).hidden_states
        m=enc.attention_mask.unsqueeze(-1)
        pooled=[((h*m).sum(1)/m.sum(1)).float().cpu().numpy() for h in hs]
        acc=[[p] for p in pooled] if acc is None else [acc[l]+[pooled[l]] for l in range(len(pooled))]
    return [np.concatenate(a,0) for a in acc]
def _d2(A,B):
    sa=(A**2).sum(1,keepdims=True); sb=(B**2).sum(1,keepdims=True); return np.clip(sa+sb.T-2*A@B.T,0,None)
def hsic(X,Y,sigma):
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    K=np.exp(-_d2(X,X)/(2*sigma**2)); L=np.exp(-_d2(Y,Y)/(2*sigma**2)); H=np.eye(n)-np.ones((n,n))/n
    return float(np.trace(K@H@L@H)/(n-1)**2)
def nhsic(X,Y,sigma):                                  # normalised HSIC in [0,1], norm-robust
    hxy=hsic(X,Y,sigma); hxx=hsic(X,X,sigma); hyy=hsic(Y,Y,sigma)
    return float(hxy/np.sqrt(hxx*hyy)) if hxx>0 and hyy>0 else 0.0
def linear_cka(X,Y):                                   # linear CKA, scale-invariant
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    Xc=X-X.mean(0,keepdims=True); Yc=Y-Y.mean(0,keepdims=True)
    num=np.linalg.norm(Yc.T@Xc)**2; den=np.linalg.norm(Xc.T@Xc)*np.linalg.norm(Yc.T@Yc)
    return float(num/den) if den>0 else 0.0
def mean_norm(X): return float(np.linalg.norm(X,axis=1).mean())
def full_pack(Lf,Lp):                                  # raw HSIC (final+per-layer) + norm-robust checks
    Xf,Xp=Lf[-1],Lp[-1]
    return {'hsic_final':{str(s):hsic(Xf,Xp,s) for s in SIGMAS},
            'hsic_per_layer':{str(s):[hsic(Lf[l],Lp[l],s) for l in range(len(Lf))] for s in SIGMAS},
            'nhsic_final':{str(s):nhsic(Xf,Xp,s) for s in SIGMAS},
            'cka_final':linear_cka(Xf,Xp),
            'norm_fair':mean_norm(Xf),'norm_priv':mean_norm(Xp)}
@torch.no_grad()
def eval_ppl(ids,seqlen=PPL_SEQLEN):
    ns=ids.numel()//seqlen; nlls=[]
    for i in range(ns):
        inp=ids[:,i*seqlen:(i+1)*seqlen].to(GEN_DEV); lg=model(inp).logits
        nlls.append(torch.nn.CrossEntropyLoss()(lg[:,:-1,:].reshape(-1,lg.size(-1)),inp[:,1:].reshape(-1)).float()*seqlen)
    return torch.exp(torch.stack(nlls).sum()/(ns*seqlen)).item()
@torch.no_grad()
def _cll(prompt, choice):
    enc=tokenizer(prompt+choice, return_tensors='pt', truncation=True, max_length=1024).to(GEN_DEV)
    plen=tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
    logits=model(**enc).logits[:, :-1, :]; tgt=enc.input_ids[:, 1:]
    logp=torch.log_softmax(logits, dim=-1).gather(2, tgt.unsqueeze(-1)).squeeze(-1)[0]
    seg=logp[plen-1:]; return float(seg.mean().item()) if seg.numel()>0 else -1e9
def mc_acc(items):
    if not items: return float('nan')
    c=0
    for it in items: c+= int(np.argmax([_cll(it['prompt'],ch) for ch in it['choices']]))==it['answer']
    return c/len(items)
def suppress_reversible(coupled):
    d=dict(model.named_parameters()); saved=[]; nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); flat=p.data.view(-1)
            saved.append((n,idx,flat[idx].clone())); flat[idx]=0.0; nz+=len(idxlist)
    return saved,nz
def restore(saved):
    d=dict(model.named_parameters())
    for n,idx,vals in saved: d[n].data.view(-1)[idx]=vals

BENCH={}
if MED_BENCH:
    try: dm=load_dataset('openlifescienceai/medmcqa',split='validation')
    except Exception: dm=load_dataset('medmcqa',split='validation',trust_remote_code=True)
    dm=dm.shuffle(seed=SEED).select(range(min(N_MEDMCQA,len(dm))))
    BENCH['MedMCQA']=[{'prompt':f"Question: {r['question']}\nAnswer:",'choices':[f" {o}" for o in (r['opa'],r['opb'],r['opc'],r['opd'])],'answer':int(r['cop'])} for r in dm]
    for s in MMLU_CLIN:
        try: ds=load_dataset('cais/mmlu',s,split='test')
        except Exception: ds=load_dataset('cais/mmlu',s,split='test',trust_remote_code=True)
        BENCH['MMLU:'+s]=[{'prompt':f"Question: {r['question']}\nAnswer:",'choices':[f" {c}" for c in r['choices']],'answer':int(r['answer'])} for r in ds]
    print('benchmarks:', {k:len(v) for k,v in BENCH.items()})
def bench_all(): return {k:mc_acc(v) for k,v in BENCH.items()} if MED_BENCH else {}
print('functions ready.')

## Step 3 — discover sets, baseline once, score every set

In [ ]:
pats={'iter':'coupled_iter_health_n*.json','global':'coupled_control_global_health_n*.json','percat':'coupled_control_percat_health_n*.json'}
SETS=[]
for variant,pat in pats.items():
    seen=set()
    for f in find_all(pat):
        m=re.search(r'_n(\d+)\.json$',f); n=int(m.group(1))
        if (variant,n) in seen: continue
        seen.add((variant,n)); SETS.append((variant,n,f))
SETS.sort(key=lambda x:(x[1],x[0]))
assert SETS, 'no coupled_iter_health_n*.json found — add iter Step-1 snapshots as input'
print('found', len(SETS), 'sets:', [(v,n) for v,n,_ in SETS])

print('baseline reps + ppl + bench...')
Lf0=reps_all(fair_q); Lp0=reps_all(priv_q); base=full_pack(Lf0,Lp0)
base_fin=base['hsic_final']; base_per=base['hsic_per_layer']
Xf0,Xp0=Lf0[-1],Lp0[-1]; del Lf0,Lp0; gc.collect()
base_ppl=eval_ppl(wt_ids); base_med=eval_ppl(med_ids) if MED_BENCH else None; base_acc=bench_all()
print('baseline HSIC(final):',{s:round(base_fin[str(s)],6) for s in SIGMAS},'| nHSIC(final):',{s:round(base['nhsic_final'][str(s)],4) for s in SIGMAS})
print('  CKA',round(base['cka_final'],4),'| norms fair',round(base['norm_fair'],3),'priv',round(base['norm_priv'],3),'| wiki ppl',round(base_ppl,4),'| pubmed ppl',(round(base_med,4) if MED_BENCH else 'off'))

rows=[]; REPS={}
for variant,n,f in SETS:
    coupled={k:[int(x) for x in v] for k,v in json.load(open(f))['coupled'].items()}
    saved,nz=suppress_reversible(coupled)
    Lf=reps_all(fair_q); Lp=reps_all(priv_q); pk=full_pack(Lf,Lp); REPS[(variant,n)]=(Lf[-1],Lp[-1]); del Lf,Lp; gc.collect()
    ppl=eval_ppl(wt_ids); med=eval_ppl(med_ids) if MED_BENCH else None; acc=bench_all(); restore(saved)
    rows.append({'variant':variant,'n':n,'coupled_zeroed':int(nz),
                 'hsic':pk['hsic_final'],'hsic_per_layer':pk['hsic_per_layer'],
                 'nhsic':pk['nhsic_final'],'cka':pk['cka_final'],
                 'norm_fair':pk['norm_fair'],'norm_priv':pk['norm_priv'],
                 'ppl':ppl,'med_ppl':med,'acc':acc})
    d400=100*(pk['hsic_final']['400']-base_fin['400'])/base_fin['400']
    print(f'  {variant:7s} n={n:<3d} zeroed {nz:5d} | HSIC@400 {d400:+.1f}% | nHSIC@400 {pk["nhsic_final"]["400"]:.4f} | CKA {pk["cka_final"]:.4f} | wiki {100*(ppl-base_ppl)/base_ppl:+.2f}% | pubmed {(100*(med-base_med)/base_med if MED_BENCH else 0):+.2f}%')

out={'model':MODEL_ID,'domain':'healthcare','sigmas':SIGMAS,
     'baseline':{'hsic':base_fin,'hsic_per_layer':base_per,'nhsic':base['nhsic_final'],'cka':base['cka_final'],
                 'norm_fair':base['norm_fair'],'norm_priv':base['norm_priv'],
                 'ppl':base_ppl,'med_ppl':base_med,'acc':base_acc},'sets':rows}
json.dump(out, open('/kaggle/working/metrics_iter_health.json','w'), indent=1)
print('saved metrics_iter_health.json')

## Step 4 — matched-budget table + bootstrap 95% CIs on HSIC drop

In [ ]:
def drop(fin,s): b=base_fin[str(s)]; return 100*(fin[str(s)]-b)/b if b else 0.0
ns=sorted(set(n for _,n,_ in SETS))
print(f'baseline: '+'  '.join(f'σ{s}={base_fin[str(s)]:.2e}' for s in SIGMAS)+f' | nHSIC@400 {base["nhsic_final"]["400"]:.4f} | CKA {base["cka_final"]:.4f} | norms f{base["norm_fair"]:.2f}/p{base["norm_priv"]:.2f} | wiki {base_ppl:.3f} | pubmed {(base_med if MED_BENCH else 0):.3f}')
mmc=lambda a: (np.mean([a['MMLU:'+s] for s in MMLU_CLIN]) if a else float('nan'))
hdr=f'{"n":>3}{"variant":>8}{"zeroed":>7} '+' '.join(f'{"dH@"+str(s):>8}' for s in SIGMAS)+f'{"nH@400":>8}{"CKA":>7}{"dwiki%":>8}{"dpubmed%":>9}{"MedMCQA":>8}{"MMLUc":>7}'
for n in ns:
    print('-'*120); print(hdr); print('-'*120)
    for variant in ['iter','global','percat']:
        r=next((x for x in rows if x['variant']==variant and x['n']==n), None)
        if not r: continue
        cells_=' '.join(f'{drop(r["hsic"],s):>+7.1f}%' for s in SIGMAS)
        med=f'{100*(r["med_ppl"]-base_med)/base_med:>+8.2f}' if MED_BENCH else f'{"-":>8}'
        mm=(f'{r["acc"]["MedMCQA"]:.3f}' if r["acc"] else '-'); mc=(f'{mmc(r["acc"]):.3f}' if r["acc"] else '-')
        print(f'{n:>3}{variant:>8}{r["coupled_zeroed"]:>7} {cells_}{r["nhsic"]["400"]:>8.4f}{r["cka"]:>7.4f}{100*(r["ppl"]-base_ppl)/base_ppl:>+7.2f}%{med:>9}{mm:>8}{mc:>7}')

if BOOT and REPS:
    n0=min(len(Xf0),len(Xp0)); rng=np.random.default_rng(0); IDX=[rng.integers(0,n0,n0) for _ in range(BOOT)]
    def _K(X,sig): return np.exp(-_d2(X[:n0],X[:n0])/(2*sig**2))
    def _hi(K,L,idx):
        Kb=K[np.ix_(idx,idx)]; Lb=L[np.ix_(idx,idx)]; n=len(idx)
        Kc=Kb-Kb.mean(0,keepdims=True)-Kb.mean(1,keepdims=True)+Kb.mean(); return (Kc*Lb).sum()/(n-1)**2
    BK={s:(_K(Xf0,s),_K(Xp0,s)) for s in SIGMAS}; ci={}
    for (variant,n),(Xf,Xp) in REPS.items():
        for s in SIGMAS:
            Kf,Lp=_K(Xf,s),_K(Xp,s); Kf0,Lp0=BK[s]; arr=np.array([100*((_hi(Kf,Lp,ix)-_hi(Kf0,Lp0,ix))/_hi(Kf0,Lp0,ix) if _hi(Kf0,Lp0,ix) else 0) for ix in IDX])
            ci[f'{variant}_n{n}_s{s}']={'med':float(np.median(arr)),'lo':float(np.percentile(arr,2.5)),'hi':float(np.percentile(arr,97.5))}
    out['bootstrap']={'B':BOOT,'drop_ci':ci}; json.dump(out, open('/kaggle/working/metrics_iter_health.json','w'), indent=1)
    print('\nHSIC drop @ σ=400, median [95% CI]:')
    for n in ns:
        for v in ['iter','global','percat']:
            k=f'{v}_n{n}_s400'
            if k in ci: print(f'  n={n:>2} {v:>7}: {ci[k]["med"]:+6.1f}%  [{ci[k]["lo"]:+6.1f}, {ci[k]["hi"]:+6.1f}]')
print('\ndone. Compare iter vs baseline across n (and vs control if present): decoupling gain per capability cost.')